In [1]:
import cv2
import matplotlib.pyplot as plt
import os

In [3]:
dataset = 'dataset'
files = os.listdir(dataset)
files

['Plant_A',
 'Plant_B',
 'Plant_C',
 'Plant_D',
 'Plant_E',
 'Plant_F',
 'Plant_G',
 'Plant_H',
 'Plant_I',
 'Plant_J',
 'plant_K',
 'Plant_L']

In [4]:
import cv2
import numpy as np
import os
from pathlib import Path

def preprocess_images_opencv(
    input_dir: str,
    output_dir: str,
    target_size: tuple = (224, 224),
    gaussian_kernel: tuple = (5, 5),
    sigma_x: float = 0,
    sharpen_strength: float = 1.0,
    image_extensions: tuple = ('.jpg')
) -> None:
    """
    Preprocess images using OpenCV:
        - Convert to grayscale
        - Resize to target_size (width, height)
        - Apply Gaussian blur
        - Sharpen the image

    Args:
        input_dir: Path to dataset folder with labelled subfolders.
        output_dir: Path where preprocessed images will be saved.
        target_size: Desired (width, height) after resizing.
        gaussian_kernel: Kernel size for Gaussian blur (odd numbers, e.g., (5,5)).
        sigma_x: Standard deviation in X direction (0 = auto from kernel size).
        sharpen_strength: Multiplier for sharpening effect (1.0 = standard kernel).
        image_extensions: Allowed image file extensions.
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)

    # Create output directory if needed
    output_path.mkdir(parents=True, exist_ok=True)

    # Sharpening kernel (Laplacian edge enhancement)
    # Standard kernel: [[0, -1, 0], [-1, 5, -1], [0, -1, 0]]
    # We'll allow strength adjustment: center = 1 + 4*strength, edges = -strength
    # For strength=1.0: center=5, edges=-1
    strength = sharpen_strength
    kernel_sharpen = np.array([
        [0, -strength, 0],
        [-strength, 1 + 4*strength, -strength],
        [0, -strength, 0]
    ], dtype=np.float32)

    # Walk through all subdirectories
    for root, dirs, files in os.walk(input_path):
        rel_path = Path(root).relative_to(input_path)
        target_subfolder = output_path / rel_path
        target_subfolder.mkdir(parents=True, exist_ok=True)

        for file in files:
            if not file.lower().endswith(image_extensions):
                continue

            img_path = Path(root) / file
            try:
                # 1. Read image (BGR by default)
                img = cv2.imread(str(img_path))
                if img is None:
                    print(f"Warning: Could not read {img_path}")
                    continue

                # 2. Convert to grayscale
                gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

                # 3. Resize to target size (width, height)
                resized = cv2.resize(gray, target_size, interpolation=cv2.INTER_LANCZOS4)

                # 4. Gaussian blur
                blurred = cv2.GaussianBlur(resized, gaussian_kernel, sigmaX=sigma_x)

                # 5. Sharpen
                sharpened = cv2.filter2D(blurred, -1, kernel_sharpen)

                # 6. Save result (preserve original file extension)
                save_path = target_subfolder / file
                cv2.imwrite(str(save_path), sharpened)
                print(f"Processed: {img_path} -> {save_path}")

            except Exception as e:
                print(f"Error processing {img_path}: {e}")

In [5]:
preprocess_images_opencv(
    input_dir = dataset,
    output_dir = 'processed',
    target_size = (224, 224),
    gaussian_kernel = (5,5),
    sigma_x = 0,
    sharpen_strength = 1.0,
    image_extensions = ('.jpg')
    )

Processed: dataset\Plant_A\video_A_frame10.jpg -> processed\Plant_A\video_A_frame10.jpg
Processed: dataset\Plant_A\video_A_frame11.jpg -> processed\Plant_A\video_A_frame11.jpg
Processed: dataset\Plant_A\video_A_frame12.jpg -> processed\Plant_A\video_A_frame12.jpg
Processed: dataset\Plant_A\video_A_frame13.jpg -> processed\Plant_A\video_A_frame13.jpg
Processed: dataset\Plant_A\video_A_frame14.jpg -> processed\Plant_A\video_A_frame14.jpg
Processed: dataset\Plant_A\video_A_frame15.jpg -> processed\Plant_A\video_A_frame15.jpg
Processed: dataset\Plant_A\video_A_frame16.jpg -> processed\Plant_A\video_A_frame16.jpg
Processed: dataset\Plant_A\video_A_frame17.jpg -> processed\Plant_A\video_A_frame17.jpg
Processed: dataset\Plant_A\video_A_frame18.jpg -> processed\Plant_A\video_A_frame18.jpg
Processed: dataset\Plant_A\video_A_frame19.jpg -> processed\Plant_A\video_A_frame19.jpg
Processed: dataset\Plant_A\video_A_frame20.jpg -> processed\Plant_A\video_A_frame20.jpg
Processed: dataset\Plant_A\video

Processed: dataset\Plant_F\Video_v_frame118.jpg -> processed\Plant_F\Video_v_frame118.jpg
Processed: dataset\Plant_G\Video_i_frame0.jpg -> processed\Plant_G\Video_i_frame0.jpg
Processed: dataset\Plant_G\Video_i_frame1.jpg -> processed\Plant_G\Video_i_frame1.jpg
Processed: dataset\Plant_G\Video_i_frame10.jpg -> processed\Plant_G\Video_i_frame10.jpg
Processed: dataset\Plant_G\Video_i_frame12.jpg -> processed\Plant_G\Video_i_frame12.jpg
Processed: dataset\Plant_G\Video_i_frame15.jpg -> processed\Plant_G\Video_i_frame15.jpg
Processed: dataset\Plant_G\Video_i_frame16.jpg -> processed\Plant_G\Video_i_frame16.jpg
Processed: dataset\Plant_G\Video_i_frame17.jpg -> processed\Plant_G\Video_i_frame17.jpg
Processed: dataset\Plant_G\Video_i_frame18.jpg -> processed\Plant_G\Video_i_frame18.jpg
Processed: dataset\Plant_G\Video_i_frame20.jpg -> processed\Plant_G\Video_i_frame20.jpg
Processed: dataset\Plant_G\Video_i_frame22.jpg -> processed\Plant_G\Video_i_frame22.jpg
Processed: dataset\Plant_G\Video_i

# splitting dataset

In [6]:
import os
import shutil
import random
from pathlib import Path
from sklearn.model_selection import train_test_split

def train_test_split_images(
    input_dir: str,
    output_dir: str,
    test_size: float = 0.2,
    train_size: float = None,
    random_state: int = 42,
    copy: bool = True
) -> None:
    """
    Split images from input_dir into train and test folders while maintaining
    the original subfolder (class label) structure.

    Args:
        input_dir: Path to folder containing labelled subfolders (e.g., 'preprocessed').
        output_dir: Destination folder where 'train' and 'test' subfolders will be created.
        test_size: Proportion of images to assign to test set (e.g., 0.2 for 20%).
        train_size: Proportion for train set (if None, uses 1 - test_size).
        random_state: Seed for reproducible splits.
        copy: If True, copies files; if False, moves them (use with caution).
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)

    # Create train/test parent directories
    train_dir = output_path / 'train'
    test_dir = output_path / 'test'
    train_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    # Get all class subfolders (only directories, ignore files)
    class_folders = [f for f in input_path.iterdir() if f.is_dir()]

    if not class_folders:
        print(f"No subfolders found in {input_dir}. Ensure your preprocessed folder contains class subfolders.")
        return

    for class_folder in class_folders:
        class_name = class_folder.name
        # Get all image files in the class folder (filter by extensions)
        image_files = [f for f in class_folder.iterdir() if f.is_file() and f.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')]

        if len(image_files) == 0:
            print(f"Warning: No images found in {class_folder}, skipping.")
            continue

        # Split indices
        train_files, test_files = train_test_split(
            image_files,
            test_size=test_size,
            train_size=train_size,
            random_state=random_state,
            shuffle=True
        )

        # Create target class subfolders inside train and test
        train_class_dir = train_dir / class_name
        test_class_dir = test_dir / class_name
        train_class_dir.mkdir(parents=True, exist_ok=True)
        test_class_dir.mkdir(parents=True, exist_ok=True)

        # Copy or move the files
        for file in train_files:
            dest = train_class_dir / file.name
            if copy:
                shutil.copy2(str(file), str(dest))
            else:
                shutil.move(str(file), str(dest))

        for file in test_files:
            dest = test_class_dir / file.name
            if copy:
                shutil.copy2(str(file), str(dest))
            else:
                shutil.move(str(file), str(dest))

        print(f"Class '{class_name}': {len(train_files)} train, {len(test_files)} test")

    print(f"Split completed. Train folder: {train_dir}, Test folder: {test_dir}")

In [7]:
train_test_split_images(
    input_dir = 'processed',
    output_dir = 'splitted',
    test_size = 0.2,
    train_size = None,
    random_state = 42,
    copy = True
)

Class 'Plant_A': 17 train, 5 test
Class 'Plant_B': 16 train, 4 test
Class 'Plant_C': 16 train, 4 test
Class 'Plant_D': 15 train, 4 test
Class 'Plant_E': 9 train, 3 test
Class 'Plant_F': 4 train, 1 test
Class 'Plant_G': 12 train, 4 test
Class 'Plant_H': 11 train, 3 test
Class 'Plant_I': 7 train, 2 test
Class 'Plant_J': 11 train, 3 test
Class 'plant_K': 12 train, 3 test
Class 'Plant_L': 6 train, 2 test
Split completed. Train folder: splitted\train, Test folder: splitted\test
